# QUALIFY lab: deduplication and temporal pricing

This notebook generates synthetic but realistic datasets for two recurring warehouse problems:

- duplicate sales exports after a source-system change
- wrong margin when using the latest supplier price instead of the price valid on the order date

The SQL examples use `QUALIFY`; the Python cells let you inspect the same logic without depending on a specific warehouse.

In [ ]:
import pandas as pd

try:
    import duckdb
except ImportError:
    duckdb = None

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
sales_rows = [
    {'order_id': 'LOT-1001', 'customer_id': 'C-204', 'game_code': 'EURO-JACK', 'gross_amount': 24.50, 'source_created_at': '2026-02-18 10:04:11', 'source_exported_at': '2026-02-18 10:15:00', 'ingestion_ts': '2026-02-18 10:17:42', 'export_batch_id': 'EXP-7701'},
    {'order_id': 'LOT-1001', 'customer_id': 'C-204', 'game_code': 'EURO-JACK', 'gross_amount': 24.50, 'source_created_at': '2026-02-18 10:04:11', 'source_exported_at': '2026-02-18 10:19:00', 'ingestion_ts': '2026-02-18 10:21:06', 'export_batch_id': 'EXP-7702'},
    {'order_id': 'LOT-1002', 'customer_id': 'C-887', 'game_code': 'MEGA-7', 'gross_amount': 12.00, 'source_created_at': '2026-02-18 10:07:43', 'source_exported_at': '2026-02-18 10:15:00', 'ingestion_ts': '2026-02-18 10:17:42', 'export_batch_id': 'EXP-7701'},
    {'order_id': 'LOT-1003', 'customer_id': 'C-522', 'game_code': 'EURO-JACK', 'gross_amount': 36.00, 'source_created_at': '2026-02-18 10:11:17', 'source_exported_at': '2026-02-18 10:15:00', 'ingestion_ts': '2026-02-18 10:17:42', 'export_batch_id': 'EXP-7701'},
    {'order_id': 'LOT-1003', 'customer_id': 'C-522', 'game_code': 'EURO-JACK', 'gross_amount': 36.00, 'source_created_at': '2026-02-18 10:11:17', 'source_exported_at': '2026-02-18 10:19:00', 'ingestion_ts': '2026-02-18 10:21:06', 'export_batch_id': 'EXP-7702'},
    {'order_id': 'LOT-1004', 'customer_id': 'C-910', 'game_code': 'CASH-5', 'gross_amount': 9.00, 'source_created_at': '2026-02-18 10:16:29', 'source_exported_at': '2026-02-18 10:19:00', 'ingestion_ts': '2026-02-18 10:21:06', 'export_batch_id': 'EXP-7702'},
    {'order_id': 'LOT-1005', 'customer_id': 'C-144', 'game_code': 'MEGA-7', 'gross_amount': 18.00, 'source_created_at': '2026-02-18 10:21:55', 'source_exported_at': '2026-02-18 10:30:00', 'ingestion_ts': '2026-02-18 10:31:40', 'export_batch_id': 'EXP-7703'},
    {'order_id': 'LOT-1005', 'customer_id': 'C-144', 'game_code': 'MEGA-7', 'gross_amount': 18.00, 'source_created_at': '2026-02-18 10:21:55', 'source_exported_at': '2026-02-18 10:34:00', 'ingestion_ts': '2026-02-18 10:35:51', 'export_batch_id': 'EXP-7704'},
    {'order_id': 'LOT-1006', 'customer_id': 'C-733', 'game_code': 'DAILY-PICK', 'gross_amount': 6.50, 'source_created_at': '2026-02-18 10:23:04', 'source_exported_at': '2026-02-18 10:30:00', 'ingestion_ts': '2026-02-18 10:31:40', 'export_batch_id': 'EXP-7703'},
    {'order_id': 'LOT-1007', 'customer_id': 'C-301', 'game_code': 'CASH-5', 'gross_amount': 11.00, 'source_created_at': '2026-02-18 10:24:48', 'source_exported_at': '2026-02-18 10:30:00', 'ingestion_ts': '2026-02-18 10:31:40', 'export_batch_id': 'EXP-7703'},
    {'order_id': 'LOT-1007', 'customer_id': 'C-301', 'game_code': 'CASH-5', 'gross_amount': 11.00, 'source_created_at': '2026-02-18 10:24:48', 'source_exported_at': '2026-02-18 10:34:00', 'ingestion_ts': '2026-02-18 10:35:51', 'export_batch_id': 'EXP-7704'},
    {'order_id': 'LOT-1008', 'customer_id': 'C-620', 'game_code': 'DAILY-PICK', 'gross_amount': 7.50, 'source_created_at': '2026-02-18 10:26:32', 'source_exported_at': '2026-02-18 10:34:00', 'ingestion_ts': '2026-02-18 10:35:51', 'export_batch_id': 'EXP-7704'}
]

sales = pd.DataFrame(sales_rows)
for col in ['source_created_at', 'source_exported_at', 'ingestion_ts']:
    sales[col] = pd.to_datetime(sales[col])

sales

In [ ]:
duplicate_groups = (
    sales.groupby('order_id', as_index=False)
    .agg(row_count=('order_id', 'size'), first_export_ts=('source_exported_at', 'min'), last_export_ts=('source_exported_at', 'max'))
    .query('row_count > 1')
    .sort_values(['row_count', 'last_export_ts'], ascending=[False, False])
)

duplicate_groups

## SQL pattern for the duplicate-export fix

```sql
SELECT
  order_id,
  customer_id,
  game_code,
  gross_amount,
  source_created_at,
  source_exported_at,
  ingestion_ts
FROM bronze_lottery_sales
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY order_id
  ORDER BY source_exported_at DESC, ingestion_ts DESC
) = 1;
```

In [ ]:
deduped_sales = (
    sales.sort_values(['order_id', 'source_exported_at', 'ingestion_ts'], ascending=[True, False, False])
    .drop_duplicates(subset=['order_id'], keep='first')
    .sort_values('order_id')
    .reset_index(drop=True)
)

print(f'Original rows: {len(sales)}')
print(f'Rows after exact DISTINCT simulation: {len(sales.drop_duplicates())}')
print(f'Rows after business-key deduplication: {len(deduped_sales)}')
deduped_sales

In [ ]:
if duckdb is None:
    print('duckdb not installed; skip SQL execution cells or run: pip install duckdb')
else:
    con = duckdb.connect()
    con.register('bronze_lottery_sales', sales)
    deduped_sales_sql = con.sql('''
        SELECT
          order_id,
          customer_id,
          game_code,
          gross_amount,
          source_created_at,
          source_exported_at,
          ingestion_ts
        FROM bronze_lottery_sales
        QUALIFY ROW_NUMBER() OVER (
          PARTITION BY order_id
          ORDER BY source_exported_at DESC, ingestion_ts DESC
        ) = 1
        ORDER BY order_id
    ''').df()
    deduped_sales_sql

In [ ]:
price_rows = [
    {'product_id': 'PRD-APPLE-BOX', 'effective_date': '2026-01-01', 'unit_cost': 13.20, 'updated_at': '2026-01-01 08:00:00'},
    {'product_id': 'PRD-APPLE-BOX', 'effective_date': '2026-02-10', 'unit_cost': 14.10, 'updated_at': '2026-02-10 07:45:00'},
    {'product_id': 'PRD-APPLE-BOX', 'effective_date': '2026-02-21', 'unit_cost': 15.30, 'updated_at': '2026-02-21 07:40:00'},
    {'product_id': 'PRD-OLIVE-OIL-5L', 'effective_date': '2026-01-15', 'unit_cost': 22.50, 'updated_at': '2026-01-15 09:10:00'},
    {'product_id': 'PRD-OLIVE-OIL-5L', 'effective_date': '2026-02-14', 'unit_cost': 24.00, 'updated_at': '2026-02-14 09:05:00'},
    {'product_id': 'PRD-OLIVE-OIL-5L', 'effective_date': '2026-02-25', 'unit_cost': 26.30, 'updated_at': '2026-02-25 09:12:00'},
    {'product_id': 'PRD-RICE-25KG', 'effective_date': '2026-01-08', 'unit_cost': 18.90, 'updated_at': '2026-01-08 06:30:00'},
    {'product_id': 'PRD-RICE-25KG', 'effective_date': '2026-02-18', 'unit_cost': 20.40, 'updated_at': '2026-02-18 06:25:00'}
]

order_rows = [
    {'order_id': 'B2B-5001', 'order_date': '2026-02-09', 'customer_name': 'Brasserie du Canal', 'product_id': 'PRD-APPLE-BOX', 'sale_price': 18.40},
    {'order_id': 'B2B-5002', 'order_date': '2026-02-12', 'customer_name': 'Resto Atlas', 'product_id': 'PRD-APPLE-BOX', 'sale_price': 18.80},
    {'order_id': 'B2B-5003', 'order_date': '2026-02-16', 'customer_name': 'Bistro Verde', 'product_id': 'PRD-OLIVE-OIL-5L', 'sale_price': 30.90},
    {'order_id': 'B2B-5004', 'order_date': '2026-02-20', 'customer_name': 'Chez Nadia', 'product_id': 'PRD-RICE-25KG', 'sale_price': 26.60},
    {'order_id': 'B2B-5005', 'order_date': '2026-02-23', 'customer_name': 'Le Comptoir Nord', 'product_id': 'PRD-APPLE-BOX', 'sale_price': 19.20},
    {'order_id': 'B2B-5006', 'order_date': '2026-02-24', 'customer_name': 'Maison du Chef', 'product_id': 'PRD-OLIVE-OIL-5L', 'sale_price': 31.40}
]

price_history = pd.DataFrame(price_rows)
orders = pd.DataFrame(order_rows)

price_history['effective_date'] = pd.to_datetime(price_history['effective_date'])
price_history['updated_at'] = pd.to_datetime(price_history['updated_at'])
orders['order_date'] = pd.to_datetime(orders['order_date'])

price_history

In [ ]:
visit_rows = [
    {'customer_id': 'C-204', 'store_id': 'SHOP-BRUSSELS-CENTRE', 'visit_date': '2026-02-03'},
    {'customer_id': 'C-204', 'store_id': 'SHOP-BRUSSELS-CENTRE', 'visit_date': '2026-02-12'},
    {'customer_id': 'C-204', 'store_id': 'SHOP-ANTWERP-NORTH', 'visit_date': '2026-02-20'},
    {'customer_id': 'C-204', 'store_id': 'SHOP-ANTWERP-NORTH', 'visit_date': '2026-02-24'},
    {'customer_id': 'C-522', 'store_id': 'SHOP-LIEGE-RIVER', 'visit_date': '2026-02-02'},
    {'customer_id': 'C-522', 'store_id': 'SHOP-LIEGE-RIVER', 'visit_date': '2026-02-09'},
    {'customer_id': 'C-522', 'store_id': 'SHOP-LIEGE-RIVER', 'visit_date': '2026-02-17'},
    {'customer_id': 'C-522', 'store_id': 'SHOP-NAMUR-GARE', 'visit_date': '2026-02-06'},
    {'customer_id': 'C-522', 'store_id': 'SHOP-NAMUR-GARE', 'visit_date': '2026-02-21'},
    {'customer_id': 'C-733', 'store_id': 'SHOP-GHENT-CENTRUM', 'visit_date': '2026-02-01'},
    {'customer_id': 'C-733', 'store_id': 'SHOP-GHENT-CENTRUM', 'visit_date': '2026-02-05'},
    {'customer_id': 'C-733', 'store_id': 'SHOP-GHENT-CENTRUM', 'visit_date': '2026-02-11'},
    {'customer_id': 'C-733', 'store_id': 'SHOP-BRUGES-MARKT', 'visit_date': '2026-02-08'},
    {'customer_id': 'C-887', 'store_id': 'SHOP-MONS-SUD', 'visit_date': '2026-02-04'},
    {'customer_id': 'C-887', 'store_id': 'SHOP-MONS-SUD', 'visit_date': '2026-02-15'},
    {'customer_id': 'C-887', 'store_id': 'SHOP-MONS-SUD', 'visit_date': '2026-02-27'},
    {'customer_id': 'C-887', 'store_id': 'SHOP-CHARLEROI-OUEST', 'visit_date': '2026-02-10'},
    {'customer_id': 'C-887', 'store_id': 'SHOP-CHARLEROI-OUEST', 'visit_date': '2026-02-18'},
    {'customer_id': 'C-887', 'store_id': 'SHOP-CHARLEROI-OUEST', 'visit_date': '2026-02-25'}
]

customer_shop_visits = pd.DataFrame(visit_rows)
customer_shop_visits['visit_date'] = pd.to_datetime(customer_shop_visits['visit_date'])
customer_shop_visits

In [ ]:
store_summary = (
    customer_shop_visits.groupby(['customer_id', 'store_id'], as_index=False)
    .agg(visit_count=('store_id', 'size'), last_visit_date=('visit_date', 'max'))
)

store_summary['dense_rank'] = (
    store_summary.groupby('customer_id')['visit_count']
    .rank(method='dense', ascending=False)
)
store_summary['rank'] = (
    store_summary.groupby('customer_id')['visit_count']
    .rank(method='min', ascending=False)
)

preferred_stores = store_summary[store_summary['dense_rank'] == 1].sort_values(['customer_id', 'store_id'])
preferred_stores

In [ ]:
pipeline_rows = [
    {'run_id': 'RUN-9001', 'pipeline_name': 'lottery-sales-bronze', 'status': 'started', 'event_ts': '2026-03-01 01:00:00', 'attempt_no': 1},
    {'run_id': 'RUN-9001', 'pipeline_name': 'lottery-sales-bronze', 'status': 'failed', 'event_ts': '2026-03-01 01:03:00', 'attempt_no': 1},
    {'run_id': 'RUN-9001', 'pipeline_name': 'lottery-sales-bronze', 'status': 'retried', 'event_ts': '2026-03-01 01:05:00', 'attempt_no': 2},
    {'run_id': 'RUN-9001', 'pipeline_name': 'lottery-sales-bronze', 'status': 'success', 'event_ts': '2026-03-01 01:08:00', 'attempt_no': 2},
    {'run_id': 'RUN-9002', 'pipeline_name': 'margin-gold-refresh', 'status': 'started', 'event_ts': '2026-03-01 02:00:00', 'attempt_no': 1},
    {'run_id': 'RUN-9002', 'pipeline_name': 'margin-gold-refresh', 'status': 'success', 'event_ts': '2026-03-01 02:06:00', 'attempt_no': 1},
    {'run_id': 'RUN-9003', 'pipeline_name': 'customer-segmentation', 'status': 'started', 'event_ts': '2026-03-01 03:00:00', 'attempt_no': 1},
    {'run_id': 'RUN-9003', 'pipeline_name': 'customer-segmentation', 'status': 'failed', 'event_ts': '2026-03-01 03:04:00', 'attempt_no': 1},
    {'run_id': 'RUN-9003', 'pipeline_name': 'customer-segmentation', 'status': 'retried', 'event_ts': '2026-03-01 03:07:00', 'attempt_no': 2},
    {'run_id': 'RUN-9003', 'pipeline_name': 'customer-segmentation', 'status': 'failed', 'event_ts': '2026-03-01 03:10:00', 'attempt_no': 2}
]

pipeline_run_log = pd.DataFrame(pipeline_rows)
pipeline_run_log['event_ts'] = pd.to_datetime(pipeline_run_log['event_ts'])
pipeline_run_log

## SQL pattern for date-valid pricing

```sql
SELECT
  o.order_id,
  o.order_date,
  o.product_id,
  o.sale_price,
  p.unit_cost,
  o.sale_price - p.unit_cost AS gross_margin
FROM b2b_orders o
JOIN supplier_price_history p
  ON o.product_id = p.product_id
 AND p.effective_date <= o.order_date
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY o.order_id
  ORDER BY p.effective_date DESC, p.updated_at DESC
) = 1;
```

In [ ]:
latest_price = (
    price_history.sort_values(['product_id', 'effective_date', 'updated_at'])
    .groupby('product_id', as_index=False)
    .tail(1)[['product_id', 'unit_cost']]
    .rename(columns={'unit_cost': 'latest_unit_cost'})
)

wrong_margin = orders.merge(latest_price, on='product_id', how='left')
wrong_margin['wrong_margin'] = wrong_margin['sale_price'] - wrong_margin['latest_unit_cost']

candidates = orders.merge(price_history, on='product_id', how='left')
candidates = candidates[candidates['effective_date'] <= candidates['order_date']].copy()
correct_margin = (
    candidates.sort_values(['order_id', 'effective_date', 'updated_at'], ascending=[True, False, False])
    .drop_duplicates(subset=['order_id'], keep='first')
    [['order_id', 'unit_cost']]
    .rename(columns={'unit_cost': 'valid_unit_cost'})
)

comparison = wrong_margin.merge(correct_margin, on='order_id', how='left')
comparison['correct_margin'] = comparison['sale_price'] - comparison['valid_unit_cost']
comparison['margin_delta'] = comparison['wrong_margin'] - comparison['correct_margin']
comparison[['order_id', 'customer_name', 'product_id', 'sale_price', 'latest_unit_cost', 'valid_unit_cost', 'wrong_margin', 'correct_margin', 'margin_delta']]

In [ ]:
if duckdb is None:
    print('duckdb not installed; skip SQL execution cells or run: pip install duckdb')
else:
    con.register('supplier_price_history', price_history)
    con.register('b2b_orders', orders)
    correct_margin_sql = con.sql('''
        SELECT
          o.order_id,
          o.order_date,
          o.customer_name,
          o.product_id,
          o.sale_price,
          p.unit_cost AS valid_unit_cost,
          o.sale_price - p.unit_cost AS gross_margin
        FROM b2b_orders o
        JOIN supplier_price_history p
          ON o.product_id = p.product_id
         AND p.effective_date <= o.order_date
        QUALIFY ROW_NUMBER() OVER (
          PARTITION BY o.order_id
          ORDER BY p.effective_date DESC, p.updated_at DESC
        ) = 1
        ORDER BY o.order_id
    ''').df()
    correct_margin_sql

In [ ]:
if duckdb is None:
    print('duckdb not installed; skip SQL execution cells or run: pip install duckdb')
else:
    con.register('customer_shop_visits', customer_shop_visits)
    preferred_stores_sql = con.sql('''
        WITH customer_store_visits AS (
          SELECT
            customer_id,
            store_id,
            COUNT(*) AS visit_count,
            MAX(visit_date) AS last_visit_date
          FROM customer_shop_visits
          GROUP BY customer_id, store_id
        )
        SELECT
          customer_id,
          store_id,
          visit_count,
          last_visit_date,
          DENSE_RANK() OVER (
            PARTITION BY customer_id
            ORDER BY visit_count DESC
          ) AS visit_rank
        FROM customer_store_visits
        QUALIFY visit_rank = 1
        ORDER BY customer_id, store_id
    ''').df()
    preferred_stores_sql

In [ ]:
latest_pipeline_status = (
    pipeline_run_log.sort_values(['run_id', 'event_ts', 'attempt_no'], ascending=[True, False, False])
    .drop_duplicates(subset=['run_id'], keep='first')
    .sort_values('run_id')
)

pipeline_run_log = pipeline_run_log.sort_values(['run_id', 'event_ts', 'attempt_no']).copy()
pipeline_run_log['previous_status'] = pipeline_run_log.groupby('run_id')['status'].shift(1)
recoveries = pipeline_run_log[
    (pipeline_run_log['previous_status'] == 'failed')
    & (pipeline_run_log['status'].isin(['retried', 'success']))
]

latest_pipeline_status, recoveries

In [ ]:
if duckdb is None:
    print('duckdb not installed; skip SQL execution cells or run: pip install duckdb')
else:
    con.register('pipeline_run_log', pipeline_run_log)
    latest_status_sql = con.sql('''
        SELECT
          run_id,
          pipeline_name,
          status,
          event_ts,
          attempt_no
        FROM pipeline_run_log
        QUALIFY ROW_NUMBER() OVER (
          PARTITION BY run_id
          ORDER BY event_ts DESC, attempt_no DESC
        ) = 1
        ORDER BY run_id
    ''').df()

    recoveries_sql = con.sql('''
        SELECT
          run_id,
          pipeline_name,
          status,
          event_ts,
          attempt_no,
          LAG(status) OVER (
            PARTITION BY run_id
            ORDER BY event_ts, attempt_no
          ) AS previous_status
        FROM pipeline_run_log
        QUALIFY previous_status = 'failed'
            AND status IN ('retried', 'success')
        ORDER BY run_id, event_ts
    ''').df()

    latest_status_sql, recoveries_sql

In [ ]:
summary = pd.DataFrame({
    'metric': ['duplicate_source_rows', 'rows_after_dedup', 'orders_with_margin_gap', 'total_margin_overstatement', 'customers_with_tied_preferred_store', 'pipeline_runs_with_recovery_event'],
    'value': [
        len(sales),
        len(deduped_sales),
        int((comparison['margin_delta'] > 0).sum()),
        round(comparison['margin_delta'].sum(), 2),
        int(preferred_stores.groupby('customer_id').size().gt(1).sum()),
        len(recoveries),
    ],
})

summary